# Data preparation for Llama
Stage 2 & 3

## A Overview

In [49]:
from pathlib import Path
import csv
import math
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
import evaluate

import torch
from datasets import Dataset
from transformers import BertForSequenceClassification, BertTokenizer, AutoTokenizer, AutoModelForSequenceClassification
from peft import PeftModel

import configuration

from src import hf_utils, setup
from src.models import bert, llama_bi

## B. Configs

In [57]:
strategy = "isolated"  # "isolated" or "crossed"
times = 2
models_path = Path("..") / "models"
datasets_path = Path("..") / "data" / "splitted" / "stage_3"
tokens_path = Path("..") / "tokens" / "stage_3" / strategy 

safety_net_path = datasets_path / "stage_3" / strategy
safety_net_path.parent.mkdir(parents=True, exist_ok=True)

# BERT
bert_model_name = "bert-base-uncased"
bert_model_path = (
    Path("..")
    / "models"
    / "stage_1"
    / f"{strategy}"
    / "BERT"
    / "w12890_o552597"
    / "1.0"
)
bert_tokenized_path = tokens_path / "BERT"/ "no_informative"
bert_tokenized_path.mkdir(parents=True, exist_ok=True)

# LLaMA
llama_model_name = "meta-llama/Llama-3.2-3B"
llama_model_path = (
    Path("..")
    / "models"
    / "stage_2"
    / f"{strategy}"
    / f"{times}_times"
    / "LLama_3_2_3B"
)

llama_tokenized_path = tokens_path / "LLaMA"/ "no_informative"
llama_tokenized_path.mkdir(parents=True, exist_ok=True)


device = setup.setup_device_with_seeds()

GPU: NVIDIA A100-SXM4-80GB
Memory allocated: 12.7 GB
Memory cached: 57.5 GB
Using device: cuda


In [16]:

df_no_informative = pd.read_csv(datasets_path / "no_informative.csv")
df_humanitarian = pd.read_csv(datasets_path / "humanitarian.csv")
print(df_no_informative.shape)
print(df_humanitarian.shape)

(54796, 5)
(84189, 5)


In [17]:
df_no_informative["informative"].value_counts()

informative
False    54796
Name: count, dtype: int64

## C. Safety Net Data

## Stage 1

In [18]:
bert_model = BertForSequenceClassification.from_pretrained(bert_model_path)
bert_tokenizer = BertTokenizer.from_pretrained(bert_model_name)
bert_model.to(device)

Loading weights: 100%|██████████| 201/201 [00:00<00:00, 6378.95it/s]

BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e-12,

In [19]:
ds_bert = Dataset.from_pandas(df_no_informative)

pre_bert_tokenized = hf_utils.tokenize(ds_bert, bert_tokenizer, bert_tokenized_path, bert.format_dataset)

Map:   0%|          | 0/54796 [00:00<?, ? examples/s]

Saving the dataset (1/1 shards): 100%|██████████| 54796/54796 [00:00<00:00, 290610.90 examples/s]


In [20]:
bert_predictions, bert_confidences = bert.predict(bert_model, pre_bert_tokenized, device, confidence_scores=True)

Predicting:: 100%|██████████| 3425/3425 [01:05<00:00, 52.35it/s]


In [21]:
bert.report_metrics(pre_bert_tokenized, bert_predictions)


Classification report:
              precision    recall  f1-score   support

           0     1.0000    0.5782    0.7327     54796
           1     0.0000    0.0000    0.0000         0

    accuracy                         0.5782     54796
   macro avg     0.5000    0.2891    0.3664     54796
weighted avg     1.0000    0.5782    0.7327     54796



/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


In [22]:
df_no_informative["stage_1_predicted"] = bert_predictions
df_no_informative["stage_1_confidence"] = bert_confidences

df_bert_failed_predictions = df_no_informative[df_no_informative["informative"] != df_no_informative["stage_1_predicted"]]
print(df_bert_failed_predictions.shape)
df_bert_failed_predictions.head()

(23112, 7)


,uid,tweet_text,informative,humanitarian_label,subset,stage_1_predicted,stage_1_confidence
3,86175,RT keyetv \RT TVsJordanSteele: #HurricanePatri...,False,NaN,disaster,1,0.677521
4,229637,RT @HighvTweet: Empire State Buildings lit up ...,False,not_humanitarian,disaster,1,0.898826
5,187252,@mikeseidel just segued to a commercial on @we...,False,NaN,disaster,1,0.792914
6,22046,BREAKING&gt;#WhiteHouse Says #USA WILLNegotiat...,False,NaN,disaster,1,0.529898
11,11068,RT NotExplained: The only known image of infam...,False,NaN,disaster,1,0.662041


## Stage 2

In [37]:
llama_base = AutoModelForSequenceClassification.from_pretrained(
    llama_model_name,
    num_labels=2,
    torch_dtype=torch.float16,  # or torch.bfloat16 depending on your hardware
 )

llama_tokenizer = AutoTokenizer.from_pretrained(llama_model_name)
# Llama 3.2 does not have a native padding token, which is required for sequence classification batching.
# Use the EOS token for padding and propagate it to model configs to avoid batch-size errors.
llama_tokenizer.pad_token = llama_tokenizer.eos_token
llama_tokenizer.padding_side = "right"
llama_base.config.pad_token_id = llama_tokenizer.pad_token_id

llama_model = PeftModel.from_pretrained(llama_base, llama_model_path)
llama_model.config.pad_token_id = llama_tokenizer.pad_token_id
llama_model.to(device)

Loading weights: 100%|██████████| 254/254 [00:08<00:00, 30.24it/s]
[transformers] LlamaForSequenceClassification LOAD REPORT from: meta-llama/Llama-3.2-3B
Key          | Status  | 
-------------+---------+-
score.weight | MISSING | 

Notes:
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


PeftModelForSequenceClassification(
  (base_model): LoraModel(
    (model): LlamaForSequenceClassification(
      (model): LlamaModel(
        (embed_tokens): Embedding(128256, 3072)
        (layers): ModuleList(
          (0-27): 28 x LlamaDecoderLayer(
            (self_attn): LlamaAttention(
              (q_proj): lora.Linear(
                (base_layer): Linear(in_features=3072, out_features=3072, bias=False)
                (lora_dropout): ModuleDict(
                  (default): Dropout(p=0.05, inplace=False)
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=3072, out_features=16, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=16, out_features=3072, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
       

In [38]:
ds_llama = Dataset.from_pandas(df_bert_failed_predictions)

In [39]:
llama_tokenized = hf_utils.tokenize(
    ds_llama, llama_tokenizer, llama_tokenized_path, llama_bi.format_dataset
)

Saving the dataset (1/1 shards): 100%|██████████| 23112/23112 [00:00<00:00, 216374.87 examples/s]


In [40]:
accuracy_metric = evaluate.load("accuracy")
precision_metric = evaluate.load("precision")
recall_metric = evaluate.load("recall")
f1_metric = evaluate.load("f1")

from torch.utils.data import DataLoader
from transformers import DataCollatorWithPadding

data_collator = DataCollatorWithPadding(tokenizer=llama_tokenizer)
dataloader = DataLoader(llama_tokenized, batch_size=8, collate_fn=data_collator)

llama_model.eval()
all_logits = []
with torch.no_grad():
    for batch in dataloader:
        batch = {
            k: v.to(device)
            for k, v in batch.items()
            if k in ("input_ids", "attention_mask")
        }
        outputs = llama_model(**batch)
        all_logits.append(outputs.logits.detach().cpu())

llama_logits = torch.cat(all_logits, dim=0)
llama_probs = torch.softmax(llama_logits, dim=-1).numpy()
llama_pred_labels = np.argmax(llama_probs, axis=-1)
llama_confidences = np.max(llama_probs, axis=-1)
test_acc = accuracy_metric.compute(predictions=llama_pred_labels, references=llama_tokenized["labels"])
test_precision = precision_metric.compute(predictions=llama_pred_labels, references=llama_tokenized["labels"])
test_recall = recall_metric.compute(predictions=llama_pred_labels, references=llama_tokenized["labels"])
test_f1 = f1_metric.compute(predictions=llama_pred_labels, references=llama_tokenized["labels"])

/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 due to no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


In [41]:
df_bert_failed_predictions["stage_2_predicted"] = llama_pred_labels
df_bert_failed_predictions["stage_2_confidence"] = llama_confidences

df_safety_net = df_bert_failed_predictions[
    df_bert_failed_predictions["informative"]
    != df_bert_failed_predictions["stage_2_predicted"]
]
print(df_safety_net.shape)
df_safety_net['humanitarian_label'] = "not_humanitarian"
df_safety_net.head()

(679, 9)


/usr/local/lib/python3.12/dist-packages/pandas/io/formats/format.py:1466: RuntimeWarning: overflow encountered in cast
  has_large_values = (abs_vals > 1e6).any()


,uid,tweet_text,informative,humanitarian_label,subset,stage_1_predicted,stage_1_confidence,stage_2_predicted,stage_2_confidence
90,114227,RT @AucklandCity_FC: VANUATU | @vanuafoot @HIR...,False,not_humanitarian,disaster,1,0.891030,1,0.613281
381,148254,"With a CERF allocation of almost $1.7 million,...",False,not_humanitarian,disaster,1,0.727841,1,0.819336
394,186049,Starting tmrw we will take pre-orders 4 our Ti...,False,not_humanitarian,disaster,1,0.834534,1,0.656738
623,39617,What is the risk assessment for people in La P...,False,not_humanitarian,disaster,1,0.657170,1,0.918457
630,38645,We persist to send message for me but we never...,False,not_humanitarian,disaster,1,0.754758,1,0.991211


## D. Merge

In [58]:
n = min(int(df_humanitarian.shape[0] * 0.1), df_safety_net.shape[0])

df_safety_net = df_safety_net.sample(n=n, random_state=setup.RANDOM_SEED).to_csv(
    safety_net_path / "safety_net.csv",
    index=False,
    quoting=csv.QUOTE_NONNUMERIC,
)

In [59]:
df_stage_3 = pd.concat([df_safety_net, df_humanitarian], ignore_index=True)
df_stage_3.to_csv(
    datasets_path / "stage_3_dataset.csv", index=False, quoting=csv.QUOTE_NONNUMERIC
)

In [60]:
df_stage_3.shape

(84189, 5)